# BÁO CÁO LÀM SẠCH VÀ PHÂN TÍCH DỮ LIỆU LƯƠNG (SALARY SURVEY)

## PHẦN 1: ĐÁNH GIÁ DATASET (CÂU 1, 2, 3)

### Câu 1: Các vấn đề của dataset
1. **Sai lệch định dạng số:** Cột `annual_salary` và `additional_monetary_comp` chứa ký tự đặc biệt ($, dấu phẩy, khoảng trắng).
2. **Dữ liệu định danh không nhất quán:** Cột `country` có nhiều biến thể như 'US', 'USA', 'United States', 'U.S.'.
3. **Tỷ lệ Null cực cao:** Cột `income_context` và `us_state` khuyết thiếu lần lượt 81% và 64%.
4. **Lỗi kiểu dữ liệu:** `timestamp` bị mixed format; các cột số đang ở dạng `object` (string).
5. **Trùng lặp:** Có các dòng trùng lặp hoàn toàn và trùng lặp theo cặp key (timestamp + job_title).
6. **Đơn vị tiền tệ hỗn hợp:** Nhiều loại currency (USD, GBP, CAD, EUR) khiến việc so sánh trực tiếp không khả thi.

### Câu 2: Hệ quả nếu dùng dataset nguyên trạng
- **Kết quả sai lệch:** Các phép tính `mean`, `sum` sẽ lỗi hoặc ra kết quả sai do outlier và ký tự lạ.
- **Phân tích bị phân mảnh:** Biểu đồ theo quốc gia sẽ hiển thị 'USA' và 'United States' là hai nhóm khác nhau.
- **Kết luận sai về thu nhập:** Không thể so sánh lương giữa một người nhận GBP và một người nhận USD nếu không quy đổi.

### Câu 3: Vấn đề nghiêm trọng nhất
- **Nghiêm trọng nhất:** Sự không nhất quán trong cột **Lương và Đơn vị tiền tệ**.
- **Tại sao:** Vì mục tiêu chính của dataset là phân tích lương. Nếu dữ liệu đầu vào là 100,000 USD và 100,000 VND mà được tính trung bình trực tiếp thành 100,000, kết quả sẽ hoàn toàn vô nghĩa và gây ra các quyết định sai lầm trong kinh doanh/nhân sự.

In [1]:
import pandas as pd
import numpy as np

# 1. Load dữ liệu
df = pd.read_csv("salary_survey_raw.csv")
df_orig = df.copy()

def missing_report(df):
    """Hàm thống kê số lượng và tỷ lệ % dữ liệu khuyết thiếu."""
    miss = df.isnull().sum()
    pct = (miss / len(df) * 100).round(2)
    return pd.DataFrame({'count': miss, 'pct': pct}).query('count > 0').sort_values('pct', ascending=False)

print("--- BÁO CÁO MISSING BAN ĐẦU ---")
print(missing_report(df))

--- BÁO CÁO MISSING BAN ĐẦU ---
                                 count    pct
income_context                    2269  81.04
us_state                          1813  64.75
city                              1562  55.79
additional_monetary_comp          1538  54.93
additional_context_on_job_title    969  34.61
race                               766  27.36
annual_salary                      391  13.96
country                            115   4.11
gender                              69   2.46


### Xác định giá trị Null:
1. **Null hợp lệ (MCAR - Missing Completely At Random):** `gender`, `race`. Những lỗi này thường do người dùng vô tình bỏ qua, không phụ thuộc vào giá trị của các biến khác.
2. **Null cần điền (MAR - Missing At Random):** `annual_salary`, `country`. Có thể suy luận từ `industry` hoặc `us_state`.
3. **Null mang thông tin (MNAR - Missing Not At Random):** `additional_monetary_comp` (54% null) và `income_context`. 
    * **Lý do:** Người có thu nhập thấp hoặc không có thưởng thường ngại khai báo hoặc mặc định bỏ trống. Nếu ta fillna bằng mean/median mà không cân nhắc, sẽ làm sai lệch nghiêm trọng phân phối thu nhập.
4. **Null cần xem xét (Investigate):** `job_title` nếu bị null thì dòng đó mất giá trị phân tích.

## PHẦN 2: QUY TRÌNH XỬ LÝ (DTYPE, DUPLICATE, MISSING)

### Câu hỏi 1: Tại sao không fillna(mean) cho lương?
Dữ liệu lương thường bị **lệch phải (Right-skewed)** do một số ít người có thu nhập cực cao (outliers). Trong trường hợp này, `mean` sẽ cao hơn hẳn so với thực tế của đại đa số người lao động. Dùng `median` ổn định hơn và phản ánh đúng mức lương "phổ biến" hơn.

In [2]:
# --- BƯỚC 1: CHUẨN HÓA KIỂU DỮ LIỆU & XỬ LÝ TRÙNG LẶP ---

# Ép kiểu thời gian (Sửa lỗi Mixed Format)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', format='mixed')

# Loại bỏ trùng lặp hoàn toàn
df = df.drop_duplicates().reset_index(drop=True)

# Chuẩn hóa lương và tiền thưởng
for col in ['annual_salary', 'additional_monetary_comp']:
    df[col] = df[col].astype(str).str.replace(r'[$,\s]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --- BƯỚC 2: XỬ LÝ MISSING VALUES THEO LOGIC ---

# 1. Xử lý Lương: Fill bằng Median theo Industry + Currency
global_median = df['annual_salary'].median()
df['annual_salary'] = df.groupby(['industry', 'currency'])['annual_salary'].transform(
    lambda x: x.fillna(x.median() if not x.dropna().empty else global_median)
)
df['annual_salary'] = df['annual_salary'].fillna(global_median)

# 2. Xử lý Country (Câu hỏi 2)
df['country'] = df['country'].str.strip().str.title()
country_mapping = {'Us': 'United States', 'Usa': 'United States', 'U.K.': 'United Kingdom', 'Uk': 'United Kingdom'}
df['country'] = df['country'].replace(country_mapping)

def impute_country(row):
    if pd.isnull(row['country']):
        return "United States" if pd.notnull(row['us_state']) else "Unknown"
    return row['country']
df['country'] = df.apply(impute_country, axis=1)

# 3. Xử lý Years of Experience (Câu hỏi 3 - Cách mất ít thông tin nhất là gán nhãn 'Unknown' thay vì drop)
df['years_of_experience_in_field'] = df['years_of_experience_in_field'].fillna('Unknown')
df['years_of_experience_overall'] = df['years_of_experience_overall'].fillna('Unknown')

# 4. Fill các cột categorical còn lại
cols_to_fix = ['additional_context_on_job_title', 'income_context', 'gender', 'race', 'city', 'us_state']
for c in cols_to_fix:
    df[c] = df[c].fillna('Unknown')
df['additional_monetary_comp'] = df['additional_monetary_comp'].fillna(0)

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

# ==============================================================================
# PHẦN 4: OUTLIER DETECTION & HANDLING (TUẦN 2)
# ==============================================================================

# 1. Phát hiện Outlier bằng phương pháp IQR (Tốt cho dữ liệu lương - lệch phải)
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    return outliers, lower, upper

outliers_iqr, low_limit, up_limit = detect_outliers_iqr(df, 'annual_salary')

# 2. Phát hiện Outlier bằng Z-score (Tốt cho phân phối chuẩn)
# Lọc bỏ NaN tạm thời để tính Z-score
temp_df = df.dropna(subset=['annual_salary']).copy()
z_scores = zscore(temp_df['annual_salary'])
outliers_z = temp_df[np.abs(z_scores) > 3]

print(f"Số lượng Outlier theo IQR: {len(outliers_iqr)}")
print(f"Số lượng Outlier theo Z-score (ngưỡng 3): {len(outliers_z)}")

# QUYẾT ĐỊNH XỬ LÝ:
# Do lương cao (CEO, Chuyên gia) là giá trị hợp lệ nhưng hiếm, ta sẽ Winsorize (clip) 
# tại percentile 1st và 99th để giảm thiểu tác động của các giá trị cực đoan.

df['annual_salary'] = df['annual_salary'].clip(
    lower=df['annual_salary'].quantile(0.01),
    upper=df['annual_salary'].quantile(0.99)
)
print("-> Đã xử lý Outlier bằng phương pháp Winsorize (Capping tại 1% và 99%).")

Số lượng Outlier theo IQR: 129
Số lượng Outlier theo Z-score (ngưỡng 3): 17
-> Đã xử lý Outlier bằng phương pháp Winsorize (Capping tại 1% và 99%).


In [4]:
# ==============================================================================
# PHẦN 5: CHUẨN HÓA CHUỖI & ENCODING (TUẦN 2 & 6)
# ==============================================================================

# 1. Pipeline chuẩn hóa chuỗi
def clean_string(s):
    if pd.isnull(s): return s
    s = str(s).lower().strip() # Lowercase & Strip
    import re
    s = re.sub(r'\s+', ' ', s) # Whitespace
    s = re.sub(r'[^a-z0-9 ]', '', s) # Special chars
    return s

text_cols = ['job_title', 'industry']
for col in text_cols:
    df[col] = df[col].apply(clean_string)

# Gom nhóm giá trị tương đồng (Ví dụ)
title_map = {
    'software engineer': 'software developer',
    'sw engineer': 'software developer',
    'swe': 'software developer'
}
df['job_title'] = df['job_title'].replace(title_map)

# 2. Encoding Categorical Variables
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Nominal Encoding (One-Hot) cho Gender
# Chúng ta dùng get_dummies đơn giản cho mục đích bài tập
df = pd.get_dummies(df, columns=['gender'], drop_first=True)

# Ordinal Encoding cho Highest Level of Education (Thứ tự có ý nghĩa)
edu_order = [
    'Unknown', 'High School', 'Some college', 'College degree', 
    'Master\'s degree', 'PhD', 'Professional degree (MD, JD, etc.)'
]
ord_enc = OrdinalEncoder(categories=[edu_order], handle_unknown='use_encoded_value', unknown_value=-1)
df['education_rank'] = ord_enc.fit_transform(df[['highest_level_of_education']].fillna('Unknown'))

# 3. Normalization (Min-Max)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[['annual_salary_scaled']] = scaler.fit_transform(df[['annual_salary']])

print("-> Đã chuẩn hóa chuỗi, thực hiện Encoding và Scaling thành công.")

-> Đã chuẩn hóa chuỗi, thực hiện Encoding và Scaling thành công.


In [5]:
# ==============================================================================
# PHẦN 6: FEATURE ENGINEERING & PIPELINE HOÀN CHỈNH (TUẦN 6)
# ==============================================================================

# 1. Feature Engineering
# Feature 1: Lương theo năm kinh nghiệm (salary_per_exp)
# Chúng ta giả định years_of_experience_overall là string, cần trích xuất số
df['years_exp_num'] = df['years_of_experience_overall'].str.extract('(\d+)').astype(float).fillna(0)
df['salary_per_year_exp'] = df['annual_salary'] / (df['years_exp_num'] + 1)

# Feature 2: Phân nhóm lương (Salary Grade)
df['salary_grade'] = pd.qcut(df['annual_salary'], q=4, labels=['Low', 'Medium', 'High', 'Elite'])

# 2. Hàm Pipeline sạch clean_data() tái sử dụng được
def clean_data(df_raw):
    """
    Pipeline làm sạch Salary Survey dataset hoàn chỉnh.
    Parameters: df_raw (DataFrame) - Dữ liệu thô ban đầu
    Returns: df_clean (DataFrame) - Dữ liệu đã qua xử lý
    """
    df_clean = df_raw.copy()
    
    # Bước 1: Xử lý trùng lặp
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)
    
    # Bước 2: Ép kiểu dữ liệu (Timestamp & Numeric)
    df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], errors='coerce', format='mixed')
    for col in ['annual_salary', 'additional_monetary_comp']:
        df_clean[col] = pd.to_numeric(df_clean[col].astype(str).str.replace(r'[$,\s]', '', regex=True), errors='coerce')
    
    # Bước 3: Điền khuyết (Sử dụng Median ngành)
    m = df_clean['annual_salary'].median()
    df_clean['annual_salary'] = df_clean.groupby('industry')['annual_salary'].transform(lambda x: x.fillna(x.median() if not x.dropna().empty else m))
    df_clean = df_clean.fillna({'additional_monetary_comp': 0, 'country': 'Unknown'})
    
    # Bước 4: Chuẩn hóa text
    df_clean['job_title'] = df_clean['job_title'].str.lower().str.strip()
    
    # Bước 5: Clip Outlier
    df_clean['annual_salary'] = df_clean['annual_salary'].clip(upper=df_clean['annual_salary'].quantile(0.99))
    
    return df_clean

print("-> Đã tạo Feature mới và xây dựng hàm clean_data(). Pipeline sẵn sàng!")

-> Đã tạo Feature mới và xây dựng hàm clean_data(). Pipeline sẵn sàng!


## PHẦN 3: DECISION LOG

### Decision Log 1: Cột 'annual_salary' — Fillna bằng Group Median
**Quan sát:** 14% hàng null. Lương có sự khác biệt rất lớn giữa các ngành (Industry) và loại tiền tệ (Currency).
**Quyết định:** Sử dụng `median` của nhóm (Industry, Currency).
**Lý do:** Giữ được tính đặc thù của ngành nghề và tránh sai lệch tỷ giá so với việc dùng global median.

### Decision Log 2: Cột 'country' — Imputation dựa trên 'us_state'
**Quan sát:** Nhiều dòng trống country nhưng lại có thông tin bang ở Mỹ (us_state).
**Quyết định:** Nếu country null và us_state không null -> gán là 'United States'.
**Lý do:** Tận dụng dữ liệu liên quan để khôi phục thông tin bị thiếu thay vì gán 'Unknown' ngay lập tức.

### Decision Log 3: Cột 'additional_monetary_comp' — Fillna(0)
**Quan sát:** Hơn 54% là null.
**Quyết định:** Điền giá trị 0.
**Lý do:** Theo logic khảo sát, nếu người dùng không điền mục thưởng thường có nghĩa là họ không nhận được khoản này, thay vì là dữ liệu bị mất (Missing Not At Random).

In [6]:
# --- BẢNG TỔNG KẾT CUỐI CÙNG ---
summary_global = pd.DataFrame({
    'Chỉ số': ['Tổng số hàng', 'Số dòng trùng lặp', 'Tổng số Null'],
    'Trước xử lý': [len(df_orig), df_orig.duplicated().sum(), df_orig.isnull().sum().sum()],
    'Sau xử lý': [len(df), df.duplicated().sum(), df.isnull().sum().sum()]
})

print("=== KẾT QUẢ SAU KHI LÀM SẠCH ===")
print(summary_global.to_string(index=False))
print("\n--- TRẠNG THÁI MISSING CUỐI CÙNG ---")
print(missing_report(df) if not missing_report(df).empty else "Chúc mừng! Không còn dữ liệu khuyết thiếu.")

=== KẾT QUẢ SAU KHI LÀM SẠCH ===
           Chỉ số  Trước xử lý  Sau xử lý
     Tổng số hàng         2800       2762
Số dòng trùng lặp           38          0
     Tổng số Null         9492          0

--- TRẠNG THÁI MISSING CUỐI CÙNG ---
Chúc mừng! Không còn dữ liệu khuyết thiếu.
